# BT4012 Fraud Analytics — E0906431

## V22: learned edge-gated transaction ranking

This single notebook contains **all model code**, including preprocessing,
temporal validation, final training and CSV generation. There are no companion
Python modules or older submission files to supply. Install `requirements.txt`
with Python3.12, place the original `train.csv`, `test.csv`, `txs_edgelist.csv` and
`sample_submission.csv` in `data/raw/`, then **Restart Kernel and Run All**.

V22 is the identified best public submission: **ROC-AUC0.95716**. The final team
private leaderboard score is0.94688, but its specific submission filename has
not been independently verified; that private score is not attributed here.

The model combines tree, tabular neural and graph neural scores, followed by
learned graph agreement gates. Only observed binary labels enter supervised
training. Whole-snapshot topology is a batch-inference assumption. Ranking
scores are not calibrated fraud probabilities.

## 1. Fixed model components

The implementation below preserves the verified V22 training logic. Base and
availability-matched graph components use distinct names so all definitions can
coexist in this notebook. These cells define models; training starts later.

The tree component uses one400-tree XGBoost model and six CatBoost models
(raw336 / time-proxy-filtered244 trees; seeds42/52/62).

In [1]:
from pathlib import Path
import copy, gc, hashlib, json, math, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import rankdata
from scipy.special import logit
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier
from threadpoolctl import threadpool_limits
import joblib, psutil, torch
from torch import nn
from torch.nn import functional as F

def base_save_json(p, x):
    Path(p).write_text(json.dumps(x, indent=2) + '\n')

def base_save(n, x):
    base_save_json(BASE_CACHE / n, x)

def base_rank(x):
    return (rankdata(x, method='average') - 0.5) / len(x)
BASE_TREE_PROFILES = {'P01': dict(family='XGB', depth=5, learning_rate=0.05, child=5, reg_lambda=1.0, reg_alpha=0.0, colsample=1.0, subsample=0.8, power=1.0, max_rounds=800), 'P06': dict(family='CAT', depth=4, learning_rate=0.06, reg_lambda=8.0, power=0.5, max_rounds=600, boosting='Ordered')}

class BaseTreeRunner:
    """Fit known labels only, caching column selections but never data copies.

    X64 preserves original float64 inputs for the exact E04 and CatBoost
    references. New profiles use float32 feature matrices to bound memory.
    Group B uses original features, S adds graph structure, and N additionally
    adds incoming/outgoing means. The drop policy removes original columns with
    absolute Spearman correlation >=0.90 with time in the fitting labels only;
    the corresponding neighbor mean columns are removed as well. Graph structure
    stays intact, and time itself is never included as a predictor.

    New configurations use four numerical threads. E04 uses exactly one, while
    the CatBoost reference uses two to reproduce the existing reference setup.
    """

    def __init__(self, X64, structure, incoming_mean, outgoing_mean, labels, times, cache_dir):
        self.X64 = np.asarray(X64)
        if self.X64.dtype != np.float64:
            raise ValueError('X64 must preserve the original float64 feature matrix.')
        self.structure = np.asarray(structure)
        self.incoming_mean = np.asarray(incoming_mean)
        self.outgoing_mean = np.asarray(outgoing_mean)
        self.y = np.asarray(labels, dtype=np.float64)
        self.times = np.asarray(times)
        n = len(self.X64)
        if self.X64.ndim != 2 or self.structure.ndim != 2:
            raise ValueError('Original and graph structure features must be matrices.')
        if len(self.structure) != n or self.incoming_mean.shape != self.X64.shape or self.outgoing_mean.shape != self.X64.shape:
            raise ValueError('All feature blocks must share the supplied feature-row order.')
        if self.y.shape != (n,) or self.times.shape != (n,):
            raise ValueError('Labels and times must align with feature rows.')
        if not np.isfinite(self.times).all():
            raise ValueError('Every supplied row requires a finite time step.')
        self.known = np.isfinite(self.y)
        if not np.isin(self.y[self.known], [0, 1]).all():
            raise ValueError('Known targets must be class 0 or 1.')
        if np.isinf(self.y).any():
            raise ValueError('Unknown labels must be NaN, never infinity.')
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.fit_log = []
        self._column_cache = {}

    def _row_indices(self, values, name):
        values = np.asarray(values)
        if values.ndim != 1 or not np.issubdtype(values.dtype, np.integer):
            raise ValueError(f'{name} must be a one-dimensional integer index array.')
        if len(values) == 0 or (values < 0).any() or (values >= len(self.y)).any():
            raise ValueError(f'{name} is empty or contains out-of-range indices.')
        return values.astype(np.int64, copy=False)

    def _columns(self, cutoff, policy, fit_idx):
        key = (int(cutoff), policy)
        if key not in self._column_cache:
            if policy == 'raw':
                removed = np.empty(0, dtype=np.int64)
            elif policy == 'drop':
                base = self.X64[fit_idx].copy()
                base[~np.isfinite(base)] = np.nan
                correlation = pd.DataFrame(base).corrwith(pd.Series(self.times[fit_idx]), method='spearman').fillna(0).abs().to_numpy()
                removed = np.flatnonzero(correlation >= 0.9)
            else:
                raise ValueError('Feature policy must be raw or drop.')
            kept = np.setdiff1d(np.arange(self.X64.shape[1]), removed, assume_unique=True)
            if len(kept) == 0:
                raise ValueError('Time-proxy removal would remove every original feature.')
            self._column_cache[key] = (kept, removed)
        return self._column_cache[key]

    def _matrix(self, rows, columns, group, dtype):
        blocks = [self.X64[np.ix_(rows, columns)]]
        if group in {'S', 'N'}:
            blocks.append(self.structure[rows])
        if group == 'N':
            blocks.extend([self.incoming_mean[np.ix_(rows, columns)], self.outgoing_mean[np.ix_(rows, columns)]])
        if group not in {'B', 'S', 'N'}:
            raise ValueError('Feature group must be B, S, or N.')
        matrix = np.concatenate(blocks, axis=1).astype(dtype, copy=False)
        matrix[~np.isfinite(matrix)] = np.nan
        return matrix

    def fit(self, config, cutoff, pred_idx, seed=42, rounds=None, early_idx=None, stage='development'):
        """Return class-1 probabilities, used rounds, and aggregate fit metadata.

        When rounds is None, supply labelled early_idx strictly later than the
        fitting cutoff. That inner validation controls early stopping only.
        Fixed-round fits never receive an evaluation set. pred_idx is separate
        from early_idx and its labels are not inspected unless explicitly reused
        as early_idx by the caller. The caller must reserve early_idx inside each
        outer fitting period, never its outer evaluation window.
        """
        started = time.perf_counter()
        cutoff = int(cutoff)
        fit_idx = np.flatnonzero(self.known & (self.times <= cutoff))
        if len(fit_idx) == 0 or len(np.unique(self.y[fit_idx])) != 2:
            raise ValueError('Fitting rows must contain known examples of both classes.')
        pred_idx = self._row_indices(pred_idx, 'pred_idx')
        if np.any(self.times[pred_idx] <= cutoff):
            raise ValueError('Prediction rows must occur after the fitting cutoff.')
        reference = config.get('reference')
        group = config.get('group', 'B')
        policy = config.get('policy', 'raw')
        if reference is not None and group != 'B':
            raise ValueError('Historical reference models use original features only.')
        if reference == 'E04':
            if policy != 'raw' or (rounds is not None and int(rounds) != 400) or early_idx is not None:
                raise ValueError('E04 requires raw original features and exactly 400 fixed rounds.')
            profile = dict(BASE_TREE_PROFILES['P01'], max_rounds=400)
            rounds = 400
            threads = 1
        elif reference == 'CAT':
            profile = dict(BASE_TREE_PROFILES['P06'], max_rounds=700)
            threads = 2
        elif reference is None:
            if config.get('profile') not in BASE_TREE_PROFILES:
                raise ValueError('Unknown prespecified tree profile.')
            profile = BASE_TREE_PROFILES[config['profile']]
            threads = 4
        else:
            raise ValueError('Unknown historical reference type.')
        early = rounds is None
        if early:
            if early_idx is None:
                raise ValueError('Early-stopping rows are required when rounds is None.')
            early_idx = self._row_indices(early_idx, 'early_idx')
            if not self.known[early_idx].all() or np.any(self.times[early_idx] <= cutoff):
                raise ValueError('Early-stopping rows must be known labels strictly after the fitting cutoff.')
            if len(np.unique(self.y[early_idx])) != 2:
                raise ValueError('Early-stopping rows must contain both target classes.')
            cap = int(profile['max_rounds'])
        else:
            if early_idx is not None:
                raise ValueError('Fixed-round fitting cannot receive early-stopping rows.')
            if int(rounds) != rounds or int(rounds) < 1:
                raise ValueError('Boosting rounds must be a positive integer.')
            cap = int(rounds)
        columns, removed = self._columns(cutoff, policy, fit_idx)
        dtype = np.float64 if reference is not None else np.float32
        a = self._matrix(fit_idx, columns, group, dtype)
        b = self._matrix(pred_idx, columns, group, dtype)
        e = self._matrix(early_idx, columns, group, dtype) if early else None
        fit_y = self.y[fit_idx].astype(np.int64)
        negative = int((fit_y == 0).sum())
        positive = int((fit_y == 1).sum())
        ratio = float((negative / positive) ** profile['power'])
        family = profile['family']
        if family == 'XGB':
            params = dict(n_estimators=cap, learning_rate=profile['learning_rate'], max_depth=profile['depth'], min_child_weight=profile['child'], subsample=profile['subsample'], colsample_bytree=profile['colsample'], reg_lambda=profile['reg_lambda'], reg_alpha=profile['reg_alpha'], objective='binary:logistic', eval_metric='auc', tree_method='hist', device='cpu', random_state=int(seed), n_jobs=threads, scale_pos_weight=ratio)
            if early:
                params['early_stopping_rounds'] = 80
            model = XGBClassifier(**params)
            with threadpool_limits(limits=threads):
                model.fit(a, fit_y, sample_weight=np.ones(len(fit_idx)) if reference == 'E04' else None, **{'eval_set': [(e, self.y[early_idx])], 'verbose': False} if early else {})
                p = model.predict_proba(b)[:, list(model.classes_).index(1)]
            used = int(model.best_iteration + 1) if early else cap
        elif family == 'LGB':
            model = LGBMClassifier(n_estimators=cap, learning_rate=profile['learning_rate'], num_leaves=profile['leaves'], max_depth=profile['depth'], min_child_samples=profile['child'], subsample=profile['subsample'], subsample_freq=1, colsample_bytree=profile['colsample'], reg_lambda=profile['reg_lambda'], reg_alpha=0.0, scale_pos_weight=ratio, objective='binary', metric='auc', random_state=int(seed), n_jobs=threads, deterministic=True, force_col_wise=True, verbosity=-1)
            with threadpool_limits(limits=threads):
                model.fit(a, fit_y, **{'eval_set': [(e, self.y[early_idx])], 'eval_metric': 'auc', 'callbacks': [early_stopping(80, first_metric_only=True, verbose=False), log_evaluation(0)]} if early else {})
                p = model.predict_proba(b)[:, list(model.classes_).index(1)]
            used = int(model.best_iteration_) if early else cap
        else:
            model = CatBoostClassifier(iterations=cap, depth=profile['depth'], learning_rate=profile['learning_rate'], l2_leaf_reg=profile['reg_lambda'], random_strength=0.5, bootstrap_type='Bayesian', bagging_temperature=0.5, boosting_type='Ordered', loss_function='Logloss', eval_metric='AUC:use_weights=false', class_weights=[1.0, ratio], random_seed=int(seed), thread_count=threads, verbose=False, allow_writing_files=False, task_type='CPU', border_count=128)
            with threadpool_limits(limits=threads):
                model.fit(a, fit_y, sample_weight=np.ones(len(fit_idx)) if reference == 'CAT' else None, **{'eval_set': (e, self.y[early_idx]), 'early_stopping_rounds': 80, 'use_best_model': True} if early else {'use_best_model': False})
                p = model.predict_proba(b)[:, list(model.classes_).index(1)]
            used = int(model.tree_count_)
        p = np.asarray(p, dtype=np.float64)
        if p.shape != (len(pred_idx),) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
            raise RuntimeError('Invalid predicted class-1 probabilities.')
        info = {'stage': stage, 'config': config.get('id', reference), 'family': family, 'reference': reference or '', 'group': group, 'policy': policy, 'profile': config.get('profile', reference), 'fit_end': cutoff, 'fit_rows': len(fit_idx), 'fit_negative': negative, 'fit_positive': positive, 'seed': int(seed), 'rounds': used, 'round_cap': cap, 'early_stopping': early, 'early_min_time': int(self.times[early_idx].min()) if early else None, 'early_max_time': int(self.times[early_idx].max()) if early else None, 'n_features': a.shape[1], 'removed': len(removed), 'removed_features': [f'feat_{j + 1}' for j in removed], 'class_weight_power': profile['power'], 'positive_multiplier': ratio, 'threads': threads, 'matrix_dtype': str(a.dtype), 'rss_gib': psutil.Process().memory_info().rss / 2 ** 30, 'seconds': round(time.perf_counter() - started, 6)}
        self.fit_log.append(info)
        base_log = pd.DataFrame(self.fit_log)
        base_log['removed_features'] = base_log['removed_features'].map(json.dumps)
        base_log.to_csv(self.cache_dir / 'tree_fit_log.csv', index=False)
        joblib.dump(model, BASE_CACHE / f'{BASE_MODEL_TAG}_model.joblib')
        np.save(BASE_CACHE / f'{BASE_MODEL_TAG}_columns.npy', columns)
        del model, a, b, e
        return (p, used, info)

### Full-snapshot graph model

A two-layer bidirectional GraphSAGE model keeps own, incoming and outgoing
features separate. It uses64 hidden units,40 epochs and seeds42/52/62.
Imputation and standardization use supplied feature rows from past snapshots.
Unlabelled rows contribute no supervised loss.

In [2]:
class BaseBidirectionalSageLayer(nn.Module):
    """Keep incoming, outgoing, and own-node information in distinct channels."""

    def __init__(self, input_width, output_width, message_dropout=0.0):
        super().__init__()
        self.own = nn.Linear(input_width, output_width)
        self.incoming = nn.Linear(input_width, output_width, bias=False)
        self.outgoing = nn.Linear(input_width, output_width, bias=False)
        self.norm = nn.LayerNorm(output_width)
        self.message_dropout = float(message_dropout)

    def forward(self, values, incoming, outgoing):
        messages = values
        if self.training and self.message_dropout > 0:
            keep = torch.rand((len(values), 1), device=values.device) >= self.message_dropout
            messages = values * keep / (1.0 - self.message_dropout)
        combined = self.own(values)
        combined = combined + self.incoming(torch.sparse.mm(incoming, messages))
        combined = combined + self.outgoing(torch.sparse.mm(outgoing, messages))
        return F.relu(self.norm(combined))

class BaseSnapshotClassifier(nn.Module):

    def __init__(self, input_width, hidden=48, dropout=0.15, kind='sage', residual=True, message_dropout=0.2):
        super().__init__()
        if kind not in {'sage', 'mlp'}:
            raise ValueError('kind must be sage or mlp')
        self.kind = kind
        self.dropout = float(dropout)
        self.residual = bool(residual and kind == 'sage')
        if kind == 'sage':
            self.first = BaseBidirectionalSageLayer(input_width, hidden, message_dropout)
            self.second = BaseBidirectionalSageLayer(hidden, hidden, message_dropout)
        else:
            self.first = nn.Sequential(nn.Linear(input_width, hidden), nn.LayerNorm(hidden), nn.ReLU())
            self.second = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.output = nn.Linear(hidden, 1)
        if self.residual:
            self.direct = nn.Sequential(nn.Linear(input_width, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, values, incoming=None, outgoing=None):
        if self.kind == 'sage':
            hidden = self.first(values, incoming, outgoing)
            hidden = F.dropout(hidden, self.dropout, self.training)
            hidden = self.second(hidden, incoming, outgoing)
        else:
            hidden = self.first(values)
            hidden = F.dropout(hidden, self.dropout, self.training)
            hidden = self.second(hidden)
        hidden = F.dropout(hidden, self.dropout, self.training)
        logits = self.output(hidden).squeeze(1)
        if self.residual:
            logits = logits + self.direct(values).squeeze(1)
        return logits

def base__snapshot_mean_tensor(adjacency, indices):
    matrix = adjacency[indices][:, indices].tocsr().astype(np.float32)
    matrix.sum_duplicates()
    matrix.eliminate_zeros()
    degree = np.asarray(matrix.sum(axis=1)).ravel()
    inverse = np.divide(1.0, degree, out=np.zeros_like(degree), where=degree > 0)
    matrix = sp.diags(inverse, format='csr') @ matrix
    coo = matrix.tocoo()
    coordinates = torch.from_numpy(np.vstack([coo.row, coo.col]).astype(np.int64))
    values = torch.from_numpy(coo.data.astype(np.float32, copy=False))
    return torch.sparse_coo_tensor(coordinates, values, size=matrix.shape).coalesce()

def base__fit_feature_statistics(graph, fit_cutoff, clip):
    values = np.asarray(graph['x'], dtype=np.float32)
    times = np.asarray(graph['node_times'])
    supplied = np.asarray(graph['supplied_features'], dtype=bool)
    fitting = supplied & (times <= fit_cutoff)
    if not fitting.any():
        raise ValueError('No supplied feature rows in the fitting period')
    fitting_values = values[fitting].copy()
    fitting_values[~np.isfinite(fitting_values)] = np.nan
    if np.isnan(fitting_values).all(axis=0).any():
        raise ValueError('An entire feature is missing in the fitting period')
    median = np.nanmedian(fitting_values, axis=0)
    fitting_values = np.where(np.isnan(fitting_values), median, fitting_values)
    mean = fitting_values.mean(axis=0, dtype=np.float64)
    std = fitting_values.std(axis=0, dtype=np.float64)
    std[std < 1e-08] = 1.0
    return {'median': median, 'mean': mean, 'std': std, 'clip': float(clip), 'fit_feature_rows': int(fitting.sum())}

def base__transform_features(values, supplied, statistics):
    values = np.asarray(values, dtype=np.float32)
    invalid = ~np.isfinite(values)
    missing = ~np.asarray(supplied, dtype=bool) | invalid.any(axis=1)
    imputed = np.where(invalid, statistics['median'], values)
    standardized = (imputed - statistics['mean']) / statistics['std']
    standardized = np.clip(standardized, -statistics['clip'], statistics['clip'])
    standardized[~np.asarray(supplied, dtype=bool)] = 0.0
    return np.column_stack([standardized, missing.astype(np.float32)]).astype(np.float32)

def base__validate_graph_model_inputs(graph, labels, fit_cutoff, prediction_indices, inner_validation_indices):
    values = np.asarray(graph['x'])
    times = np.asarray(graph['node_times'])
    supplied = np.asarray(graph['supplied_features'])
    labels = np.asarray(labels, dtype=float)
    n = len(values)
    if values.ndim != 2 or times.shape != (n,) or supplied.shape != (n,) or (labels.shape != (n,)):
        raise ValueError('Graph arrays do not align')
    if not np.isfinite(times).all() or not (times > 0).all():
        raise ValueError('Only anchored positive-time nodes may enter the model')
    for name in ['incoming', 'outgoing']:
        adjacency = graph[name]
        if not sp.isspmatrix_csr(adjacency) or adjacency.shape != (n, n):
            raise ValueError(f'{name} must be an aligned scipy CSR matrix')
        coo = adjacency.tocoo(copy=False)
        if not np.all(times[coo.row] == times[coo.col]):
            raise ValueError('Cross-time message passing is prohibited')
        if not np.isfinite(coo.data).all() or (coo.data < 0).any():
            raise ValueError('Adjacency must have finite nonnegative weights')
    finite_labels = np.isfinite(labels)
    if not set(np.unique(labels[finite_labels])).issubset({0.0, 1.0}):
        raise ValueError('Known labels must be zero or one')
    fitting = finite_labels & supplied & (times <= fit_cutoff)
    if len(np.unique(labels[fitting])) != 2:
        raise ValueError('Both supervised classes must occur in fitting rows')
    if np.any(finite_labels & ~supplied):
        raise ValueError('Feature-missing edge-only nodes cannot have supplied labels')
    for name, indices in [('prediction', prediction_indices), ('inner validation', inner_validation_indices)]:
        if indices is None:
            continue
        indices = np.asarray(indices)
        if indices.ndim != 1 or not np.issubdtype(indices.dtype, np.integer):
            raise ValueError(f'{name} indices must be a one-dimensional integer array')
        if len(np.unique(indices)) != len(indices) or (indices < 0).any() or (indices >= n).any():
            raise ValueError(f'Invalid or duplicate {name} indices')
        if not np.all(times[indices] > fit_cutoff):
            raise ValueError(f'{name} must be strictly later than the fitting cutoff')
    if inner_validation_indices is not None:
        indices = np.asarray(inner_validation_indices, dtype=int)
        if not finite_labels[indices].all() or len(np.unique(labels[indices])) != 2:
            raise ValueError('Early stopping requires supplied labels from both classes')
    return (labels, fitting)

def base_fit_predict_graph_model(graph, labels, fit_cutoff, prediction_indices, config, seed=42, epochs=None, inner_validation_indices=None):
    """Train on labelled fitting snapshots and predict later snapshots.

    With epochs=None, inner_validation_indices is mandatory and controls early
    stopping. With a fixed epoch count, no validation labels affect fitting.
    config keys: kind, hidden, dropout, residual, learning_rate, weight_decay,
    class_power, max_epochs, patience, min_epochs, snapshot_weighting, clip, message_dropout,
    threads, verbose. No external fit/prediction cache is read or written.
    """
    started = time.perf_counter()
    defaults = {'kind': 'sage', 'hidden': 48, 'dropout': 0.15, 'residual': True, 'learning_rate': 0.003, 'weight_decay': 0.0001, 'class_power': 0.5, 'max_epochs': 60, 'patience': 10, 'min_epochs': 8, 'snapshot_weighting': 'rows', 'clip': 8.0, 'threads': 2, 'verbose': False, 'message_dropout': 0.2}
    if set(config) - set(defaults):
        raise ValueError(f'Unrecognized graph model settings: {sorted(set(config) - set(defaults))}')
    cfg = {**defaults, **config}
    if cfg['snapshot_weighting'] not in {'rows', 'equal_time'}:
        raise ValueError('snapshot_weighting must be rows or equal_time')
    if cfg['kind'] not in {'sage', 'mlp'}:
        raise ValueError('kind must be sage or mlp')
    if not 0 <= cfg['message_dropout'] < 1 or not 0 <= cfg['dropout'] < 1:
        raise ValueError('Dropout rates must lie in [0,1)')
    if epochs is None and inner_validation_indices is None:
        raise ValueError('Inner validation is required when selecting the epoch count')
    if epochs is not None and inner_validation_indices is not None:
        raise ValueError('Fixed-epoch fitting cannot receive inner validation indices')
    total_epochs = int(cfg['max_epochs'] if epochs is None else epochs)
    if total_epochs < 1:
        raise ValueError('At least one training epoch is required')
    prediction_indices = np.asarray(prediction_indices, dtype=np.int64)
    if inner_validation_indices is not None:
        inner_validation_indices = np.asarray(inner_validation_indices, dtype=np.int64)
    labels, fitting = base__validate_graph_model_inputs(graph, labels, fit_cutoff, prediction_indices, inner_validation_indices)
    torch.set_num_threads(int(cfg['threads']))
    torch.manual_seed(int(seed))
    rng = np.random.default_rng(int(seed))
    statistics = base__fit_feature_statistics(graph, fit_cutoff, cfg['clip'])
    node_times = np.asarray(graph['node_times'])
    supplied = np.asarray(graph['supplied_features'], dtype=bool)
    fit_steps = np.unique(node_times[fitting])
    needed_steps = set(fit_steps.tolist()) | set(node_times[prediction_indices].tolist())
    if inner_validation_indices is not None:
        needed_steps |= set(node_times[inner_validation_indices].tolist())
    snapshots = {}
    fit_counts = {}
    for step in sorted(needed_steps):
        indices = np.flatnonzero(node_times == step)
        if cfg['kind'] == 'mlp':
            indices = indices[supplied[indices]]
        transformed = base__transform_features(np.asarray(graph['x'])[indices], supplied[indices], statistics)
        local_fit = np.flatnonzero(fitting[indices])
        item = {'indices': indices, 'x': torch.from_numpy(transformed), 'supervised': torch.from_numpy(local_fit)}
        if len(local_fit):
            item['y'] = torch.from_numpy(labels[indices[local_fit]].astype(np.float32))
            fit_counts[int(step)] = len(local_fit)
        if cfg['kind'] == 'sage':
            item['incoming'] = base__snapshot_mean_tensor(graph['incoming'], indices)
            item['outgoing'] = base__snapshot_mean_tensor(graph['outgoing'], indices)
        snapshots[int(step)] = item
    fit_indices = np.flatnonzero(fitting)
    class_weights = np.ones(len(fit_indices), dtype=float)
    if cfg['snapshot_weighting'] == 'equal_time':
        class_weights = np.array([1.0 / fit_counts[int(step)] for step in node_times[fit_indices]])
    positive = class_weights[labels[fit_indices] == 1].sum()
    negative = class_weights[labels[fit_indices] == 0].sum()
    positive_multiplier = float((negative / positive) ** float(cfg['class_power']))
    positive_tensor = torch.tensor(positive_multiplier, dtype=torch.float32)
    model = BaseSnapshotClassifier(np.asarray(graph['x']).shape[1] + 1, int(cfg['hidden']), cfg['dropout'], cfg['kind'], cfg['residual'], cfg['message_dropout'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(cfg['learning_rate']), weight_decay=float(cfg['weight_decay']))
    mean_count = float(np.mean(list(fit_counts.values())))
    history = []
    best_auc = -np.inf
    best_epoch = 0
    best_state = None
    stale = 0

    def forward_snapshot(item):
        return model(item['x'], item.get('incoming'), item.get('outgoing'))

    def predict_indices(indices):
        probability = np.empty(len(indices), dtype=np.float64)
        model.eval()
        with torch.no_grad():
            for step in np.unique(node_times[indices]):
                positions = np.flatnonzero(node_times[indices] == step)
                requested = indices[positions]
                item = snapshots[int(step)]
                local = np.searchsorted(item['indices'], requested)
                if (local >= len(item['indices'])).any() or not np.array_equal(item['indices'][local], requested):
                    raise ValueError('Prediction row is absent from its snapshot')
                logits = forward_snapshot(item)
                probability[positions] = torch.sigmoid(logits[torch.from_numpy(local)]).numpy()
        return probability
    for epoch in range(1, total_epochs + 1):
        model.train()
        total_loss = 0.0
        for step in rng.permutation(fit_steps):
            item = snapshots[int(step)]
            optimizer.zero_grad(set_to_none=True)
            logits = forward_snapshot(item)[item['supervised']]
            loss = F.binary_cross_entropy_with_logits(logits, item['y'], pos_weight=positive_tensor)
            if cfg['snapshot_weighting'] == 'rows':
                loss = loss * (fit_counts[int(step)] / mean_count)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += float(loss.detach())
        record = {'epoch': epoch, 'mean_training_loss': total_loss / len(fit_steps)}
        if inner_validation_indices is not None:
            validation_probability = predict_indices(inner_validation_indices)
            auc = float(roc_auc_score(labels[inner_validation_indices], validation_probability))
            record['inner_auc'] = auc
            if auc > best_auc + 1e-06:
                best_auc, best_epoch = (auc, epoch)
                best_state = copy.deepcopy(model.state_dict())
                stale = 0
            else:
                stale += 1
            history.append(record)
            if cfg['verbose']:
                print(f'{cfg['kind']} epoch {epoch}: inner AUC {auc:.6f}', flush=True)
            if epoch >= int(cfg['min_epochs']) and stale >= int(cfg['patience']):
                break
        else:
            history.append(record)
    if inner_validation_indices is not None:
        model.load_state_dict(best_state)
        used_epochs = best_epoch
    else:
        used_epochs = total_epochs
    probability = predict_indices(prediction_indices)
    if not np.isfinite(probability).all() or not ((probability >= 0) & (probability <= 1)).all():
        raise AssertionError('Graph model returned invalid probabilities')
    info = {'kind': cfg['kind'], 'seed': int(seed), 'fit_cutoff': int(fit_cutoff), 'fitting_labels': int(fitting.sum()), 'fit_feature_rows': statistics['fit_feature_rows'], 'fitting_snapshots': len(fit_steps), 'positive_multiplier': positive_multiplier, 'epochs_executed': len(history), 'selected_epochs': used_epochs, 'inner_best_auc': best_auc if inner_validation_indices is not None else None, 'parameters': sum((parameter.numel() for parameter in model.parameters())), 'seconds': time.perf_counter() - started, 'config': cfg, 'history': history, 'device': 'CPU', 'transformation': 'fitting-only imputation/standardization plus missing-row flag'}
    np.savez_compressed(BASE_CACHE / f'{BASE_MODEL_TAG}_weights.npz', **{k: v.detach().numpy() for k, v in model.state_dict().items()})
    joblib.dump(statistics, BASE_CACHE / f'{BASE_MODEL_TAG}_statistics.joblib')
    return (probability, used_epochs, info)

### Tabular neural rank component

The eight-head shared MLP uses width128, past-only quantile preprocessing,
24 fixed epochs and seeds42/52/62. Strong time-proxy features are removed using
only the fitting period. This is a custom shared-parameter ensemble, not an
exact reproduction of TabM.

In [3]:
class BaseTemporalFeatureTransform:
    """Fit raw, drop90, stable, or nuisance3 without target or future inputs.

    Pass only the fitting partition's labelled feature rows to fit. Targets are
    deliberately absent from this interface. transform uses frozen state and
    never learns from a prediction batch, its time steps, or future observations.

    drop90 removes features whose absolute fitting-row Spearman correlation with
    time is >=0.90. stable removes features at >=0.70 or whose between-snapshot
    median IQR exceeds ten times their typical within-snapshot IQR. When the
    denominator is zero, a positive numerator gives infinity; zero/zero is zero.
    Thus constant sparse features are not discarded merely for having zero IQR.

    nuisance3 estimates up to three major changing directions from equally
    weighted fitting-snapshot median vectors after fitting-only standardization.
    It subtracts each row's projection onto those directions. This removes linear
    nuisance directions and can also remove useful signal, so it is a candidate
    representation to validate, not a guaranteed invariant fraud representation.
    All 165 anonymous coordinates are considered; no feature meaning or ordering
    partition is assumed. The direction count and thresholds are prespecified.
    """

    def __init__(self, policy='raw'):
        if policy not in {'raw', 'drop90', 'stable', 'nuisance3'}:
            raise ValueError('Unknown temporal feature policy.')
        self.policy = policy
        self.fitted_ = False

    @staticmethod
    def _values(X):
        values = np.asarray(X, dtype=np.float64)
        if values.ndim != 2 or values.shape[0] == 0 or values.shape[1] == 0:
            raise ValueError('Features must form a nonempty two-dimensional matrix.')
        values = values.copy()
        values[~np.isfinite(values)] = np.nan
        return values

    def fit(self, X, times):
        values = self._values(X)
        times = np.asarray(times)
        if times.shape != (len(values),) or not np.isfinite(times).all():
            raise ValueError('Fitting times must be finite and align with fitting rows.')
        if np.isnan(values).all(axis=0).any():
            raise ValueError('Cannot estimate a feature absent from every fitting row.')
        self.n_features_in_ = values.shape[1]
        self.kept_ = np.arange(self.n_features_in_)
        self.removed_ = np.empty(0, dtype=np.int64)
        self.time_spearman_ = np.zeros(self.n_features_in_)
        self.between_within_ratio_ = np.zeros(self.n_features_in_)
        self.components_ = np.empty((0, self.n_features_in_))
        frame = pd.DataFrame(values)
        if self.policy in {'drop90', 'stable'}:
            self.time_spearman_ = frame.rank().corrwith(pd.Series(times).rank()).fillna(0).to_numpy()
            threshold = 0.9 if self.policy == 'drop90' else 0.7
            remove = np.abs(self.time_spearman_) >= threshold
            if self.policy == 'stable':
                grouped = frame.groupby(times)
                medians = grouped.median()
                between = (medians.quantile(0.75) - medians.quantile(0.25)).to_numpy()
                within = (grouped.quantile(0.75) - grouped.quantile(0.25)).median(axis=0).to_numpy()
                ratio = np.divide(between, within, out=np.zeros_like(between), where=within > 0)
                ratio[(within == 0) & (between > 0)] = np.inf
                self.between_within_ratio_ = ratio
                remove |= ratio > 10.0
            self.removed_ = np.flatnonzero(remove)
            self.kept_ = np.flatnonzero(~remove)
            if len(self.kept_) == 0:
                raise ValueError('The fixed temporal stability rule removed every feature.')
        elif self.policy == 'nuisance3':
            if len(np.unique(times)) < 2:
                raise ValueError('Temporal nuisance estimation requires at least two fitting snapshots.')
            self.imputation_medians_ = np.nanmedian(values, axis=0)
            filled = np.where(np.isfinite(values), values, self.imputation_medians_)
            self.mean_ = filled.mean(axis=0)
            self.scale_ = filled.std(axis=0, ddof=0)
            self.scale_[self.scale_ <= np.finfo(np.float64).eps] = 1.0
            standardized = (filled - self.mean_) / self.scale_
            snapshot_medians = pd.DataFrame(standardized).groupby(times).median().to_numpy()
            self.direction_center_ = snapshot_medians.mean(axis=0)
            centered = snapshot_medians - self.direction_center_
            _, singular_values, right = np.linalg.svd(centered, full_matrices=False)
            tolerance = max(centered.shape) * np.finfo(np.float64).eps * singular_values.max(initial=0)
            directions = min(3, int((singular_values > tolerance).sum()))
            self.components_ = right[:directions]
            total = (singular_values ** 2).sum()
            self.snapshot_variance_removed_ = float((singular_values[:directions] ** 2).sum() / total) if total > 0 else 0.0
        self.info_ = {'policy': self.policy, 'fit_rows': len(values), 'fit_min_time': int(np.min(times)), 'fit_max_time': int(np.max(times)), 'fit_snapshots': int(len(np.unique(times))), 'input_features': self.n_features_in_, 'output_features': len(self.kept_), 'removed_features': [f'feat_{j + 1}' for j in self.removed_], 'time_spearman_threshold': 0.9 if self.policy == 'drop90' else 0.7 if self.policy == 'stable' else None, 'between_within_ratio_threshold': 10.0 if self.policy == 'stable' else None, 'nuisance_directions': len(self.components_), 'snapshot_variance_removed': getattr(self, 'snapshot_variance_removed_', None), 'prediction_batch_statistics_used': False}
        self.fitted_ = True
        return self

    def transform(self, X):
        if not self.fitted_:
            raise RuntimeError('Fit the temporal feature transform first.')
        values = self._values(X)
        if values.shape[1] != self.n_features_in_:
            raise ValueError('Prediction feature width differs from the fitting width.')
        if self.policy == 'nuisance3':
            filled = np.where(np.isfinite(values), values, self.imputation_medians_)
            standardized = (filled - self.mean_) / self.scale_
            centered = standardized - self.direction_center_
            result = standardized - centered @ self.components_.T @ self.components_
        else:
            result = values[:, self.kept_]
        return result.astype(np.float32)

    def fit_transform(self, X, times):
        return self.fit(X, times).transform(X)

class BaseFitOnlyNumericalTransform:
    """Fit imputation and scaling on the fitting matrix, then transform copies."""

    def __init__(self, method='quantile', seed=42, n_quantiles=256, clip=8.0):
        if method not in {'standard', 'quantile'}:
            raise ValueError('Preprocessing must be standard or quantile.')
        if int(n_quantiles) < 2 or not np.isfinite(clip) or clip <= 0:
            raise ValueError('Quantile count and clipping threshold must be positive.')
        self.method = method
        self.seed = int(seed)
        self.n_quantiles = int(n_quantiles)
        self.clip = float(clip)

    def fit(self, values):
        values = np.asarray(values, dtype=np.float64)
        if values.ndim != 2 or min(values.shape) < 1:
            raise ValueError('Fitting features must be a nonempty numerical matrix.')
        clean = values.copy()
        clean[~np.isfinite(clean)] = np.nan
        if np.isnan(clean).all(axis=0).any():
            raise ValueError('An entire fitting feature is missing; inspect it before fitting.')
        self.median_ = np.nanmedian(clean, axis=0)
        clean = np.where(np.isnan(clean), self.median_, clean)
        self.n_features_in_ = clean.shape[1]
        self.fit_rows_ = len(clean)
        if self.method == 'standard':
            self.scaler_ = StandardScaler()
        else:
            self.scaler_ = QuantileTransformer(n_quantiles=min(self.n_quantiles, len(clean)), output_distribution='normal', subsample=max(100000, len(clean)), random_state=self.seed, copy=True)
        self.scaler_.fit(clean)
        state = hashlib.sha256(self.median_.tobytes())
        for name in ['mean_', 'scale_', 'quantiles_', 'references_']:
            if hasattr(self.scaler_, name):
                state.update(np.asarray(getattr(self.scaler_, name)).tobytes())
        self.state_sha256_ = state.hexdigest()
        return self

    def transform(self, values):
        values = np.asarray(values, dtype=np.float64)
        if values.ndim != 2 or values.shape[1] != self.n_features_in_ or len(values) == 0:
            raise ValueError('Prediction features must align with fitting columns.')
        clean = np.where(np.isfinite(values), values, self.median_)
        result = self.scaler_.transform(clean)
        result = np.clip(result, -self.clip, self.clip).astype(np.float32)
        if not np.isfinite(result).all():
            raise ValueError('Numerical preprocessing returned a nonfinite value.')
        return np.ascontiguousarray(result)

class BaseSharedEnsembleLinear(nn.Module):
    """A shared dense weight matrix with member-specific input/output scales."""

    def __init__(self, input_width, output_width, members):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(output_width, input_width))
        self.input_scale = nn.Parameter(torch.empty(members, input_width))
        self.output_scale = nn.Parameter(torch.empty(members, output_width))
        self.bias = nn.Parameter(torch.zeros(members, output_width))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        with torch.no_grad():
            self.input_scale.copy_(torch.randint(0, 2, self.input_scale.shape) * 2 - 1)
            self.output_scale.copy_(torch.randint(0, 2, self.output_scale.shape) * 2 - 1)

    def forward(self, values):
        return F.linear(values * self.input_scale, self.weight) * self.output_scale + self.bias

class BaseParameterEfficientMLP(nn.Module):
    """Two shared hidden layers and a separate classification head per member."""

    def __init__(self, n_features, k=8, width=64, dropout=0.05):
        super().__init__()
        self.k = int(k)
        self.first = BaseSharedEnsembleLinear(n_features, width, self.k)
        self.second = BaseSharedEnsembleLinear(width, width, self.k)
        self.dropout = float(dropout)
        self.output_weight = nn.Parameter(torch.empty(self.k, width))
        self.output_bias = nn.Parameter(torch.zeros(self.k))
        nn.init.normal_(self.output_weight, mean=0.0, std=1.0 / math.sqrt(width))

    def forward(self, values):
        hidden = values[:, None, :].expand(-1, self.k, -1)
        hidden = F.relu(self.first(hidden))
        hidden = F.dropout(hidden, p=self.dropout, training=self.training)
        hidden = F.relu(self.second(hidden))
        hidden = F.dropout(hidden, p=self.dropout, training=self.training)
        return (hidden * self.output_weight).sum(dim=-1) + self.output_bias

def base__binary_targets(values, rows, name):
    values = np.asarray(values, dtype=np.float32)
    if values.shape != (rows,) or not np.isfinite(values).all():
        raise ValueError(f'{name} must contain one finite known target per row.')
    if set(np.unique(values)) != {0.0, 1.0}:
        raise ValueError(f'{name} must contain both classes, encoded as 0 and 1.')
    return values

def base_fit_predict_ensemble(X_fit, y_fit, X_pred, config=None, seed=42, epochs=None, X_inner=None, y_inner=None):
    """Return (class-1 probability, selected epochs, aggregate fit metadata).

    epochs=None requires an explicitly supplied inner temporal validation set.
    Fixed-epoch fitting rejects either inner argument. Prediction labels are not
    an API input. Imputation/scaling fit only X_fit in both modes. The caller must
    supply temporal partitions and keep inner rows inside the outer fitting
    period. Each ensemble head receives its own binary loss; inference averages
    head probabilities, not logits. Unknown labels cannot enter supervised loss.
    """
    started = time.perf_counter()
    defaults = {'k': 8, 'width': 64, 'dropout': 0.05, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 512, 'preprocessing': 'quantile', 'class_power': 0.5, 'max_epochs': 80, 'patience': 12, 'min_epochs': 8, 'threads': 2, 'n_quantiles': 256, 'clip': 8.0, 'verbose': False}
    config = {} if config is None else dict(config)
    if set(config) - set(defaults):
        raise ValueError(f'Unknown ensemble settings: {sorted(set(config) - set(defaults))}')
    cfg = {**defaults, **config}
    for key in ['k', 'width', 'batch_size', 'max_epochs', 'patience', 'min_epochs', 'threads']:
        if int(cfg[key]) != cfg[key] or cfg[key] < 1:
            raise ValueError(f'{key} must be a positive integer.')
    if not 0 <= cfg['dropout'] < 1 or not 0 <= cfg['class_power'] <= 1:
        raise ValueError('Dropout and class weighting power are outside their allowed ranges.')
    if cfg['learning_rate'] <= 0 or cfg['weight_decay'] < 0:
        raise ValueError('Optimizer settings must be positive/nonnegative.')
    early = epochs is None
    if early and (X_inner is None or y_inner is None):
        raise ValueError('An inner temporal validation set is required to select epochs.')
    if not early and (X_inner is not None or y_inner is not None):
        raise ValueError('Fixed-epoch fits cannot receive inner validation features or labels.')
    total_epochs = cfg['max_epochs'] if early else epochs
    if int(total_epochs) != total_epochs or total_epochs < 1:
        raise ValueError('Epoch count must be a positive integer.')
    total_epochs = int(total_epochs)
    X_fit = np.asarray(X_fit)
    if X_fit.ndim != 2:
        raise ValueError('Fitting features must be a numerical matrix.')
    y = base__binary_targets(y_fit, len(X_fit), 'Fitting targets')
    torch.set_num_threads(int(cfg['threads']))
    torch.manual_seed(int(seed))
    batch_rng = torch.Generator(device='cpu').manual_seed(int(seed) + 1009)
    transform = BaseFitOnlyNumericalTransform(cfg['preprocessing'], seed, cfg['n_quantiles'], cfg['clip']).fit(X_fit)
    fit_tensor = torch.from_numpy(transform.transform(X_fit))
    pred_tensor = torch.from_numpy(transform.transform(X_pred))
    target_tensor = torch.from_numpy(y)
    if early:
        inner_tensor = torch.from_numpy(transform.transform(X_inner))
        inner_targets = base__binary_targets(y_inner, len(inner_tensor), 'Inner targets')
    else:
        inner_tensor = inner_targets = None
    ratio = float(((y == 0).sum() / (y == 1).sum()) ** cfg['class_power'])
    positive_weight = torch.tensor(ratio, dtype=torch.float32)
    model = BaseParameterEfficientMLP(X_fit.shape[1], int(cfg['k']), int(cfg['width']), cfg['dropout'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(cfg['learning_rate']), weight_decay=float(cfg['weight_decay']))
    history = []
    best_auc, best_epoch, best_state, stale = (-np.inf, 0, None, 0)
    observed_rss = psutil.Process().memory_info().rss / 2 ** 30

    def probability(values):
        model.eval()
        result = []
        with torch.no_grad():
            for start in range(0, len(values), int(cfg['batch_size'])):
                logits = model(values[start:start + int(cfg['batch_size'])])
                result.append(torch.sigmoid(logits).mean(dim=1).numpy())
        scores = np.concatenate(result).astype(np.float64)
        if not np.isfinite(scores).all() or not ((scores >= 0) & (scores <= 1)).all():
            raise RuntimeError('The neural ensemble returned invalid probabilities.')
        return scores
    for epoch in range(1, total_epochs + 1):
        model.train()
        order = torch.randperm(len(y), generator=batch_rng)
        total_loss = 0.0
        for start in range(0, len(y), int(cfg['batch_size'])):
            indices = order[start:start + int(cfg['batch_size'])]
            optimizer.zero_grad(set_to_none=True)
            logits = model(fit_tensor[indices])
            targets = target_tensor[indices, None].expand_as(logits)
            loss = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=positive_weight)
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite neural training loss.')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += float(loss.detach()) * len(indices)
        record = {'epoch': epoch, 'training_loss': total_loss / len(y)}
        observed_rss = max(observed_rss, psutil.Process().memory_info().rss / 2 ** 30)
        if early:
            auc = float(roc_auc_score(inner_targets, probability(inner_tensor)))
            record['inner_auc'] = auc
            if auc > best_auc + 1e-06:
                best_auc, best_epoch = (auc, epoch)
                best_state = copy.deepcopy(model.state_dict())
                stale = 0
            else:
                stale += 1
        history.append(record)
        if cfg['verbose']:
            print(f'Custom ensemble epoch {epoch}: {record}', flush=True)
        if early and epoch >= int(cfg['min_epochs']) and (stale >= int(cfg['patience'])):
            break
    if early:
        if best_state is None:
            raise RuntimeError('Inner validation did not produce a finite best checkpoint.')
        model.load_state_dict(best_state)
        used = best_epoch
    else:
        used = total_epochs
    prediction = probability(pred_tensor)
    info = {'architecture': 'custom two-layer BatchEnsemble-style MLP; not exact TabM', 'seed': int(seed), 'fit_rows': len(y), 'prediction_rows': len(prediction), 'n_features': X_fit.shape[1], 'parameters': sum((p.numel() for p in model.parameters())), 'positive_multiplier': ratio, 'preprocessing': cfg['preprocessing'], 'transform_fit_rows': transform.fit_rows_, 'transform_sha256': transform.state_sha256_, 'epochs_executed': len(history), 'selected_epochs': int(used), 'inner_best_auc': float(best_auc) if early else None, 'seconds': time.perf_counter() - started, 'rss_gib': observed_rss, 'device': 'CPU', 'config': cfg, 'history': history}
    np.savez_compressed(BASE_CACHE / f'{BASE_MODEL_TAG}_weights.npz', **{k: v.detach().numpy() for k, v in model.state_dict().items()})
    joblib.dump(transform, BASE_CACHE / f'{BASE_MODEL_TAG}_transform.joblib')
    return (prediction, int(used), info)

def base_log(info, tag):
    record = {k: v for k, v in info.items() if k not in ['history']}
    record['tag'] = tag
    BASE_logs.append(record)
    base_save('fit_log.json', BASE_logs)
    print(tag, 'seconds', round(info['seconds'], 2), flush=True)

### Availability-matched graph component

This second GraphSAGE representation fits quantiles on observed target rows in
the past. Nodes without available features have zeroed transformed features and
an explicit availability flag. It uses40 epochs and the same three seeds.

In [4]:
from pathlib import Path
import gc, hashlib, json, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import rankdata
from sklearn.preprocessing import QuantileTransformer
from threadpoolctl import threadpool_limits
import psutil, torch
from torch import nn
from torch.nn import functional as F

class TargetBidirectionalSageLayer(nn.Module):
    """Keep incoming, outgoing, and own-node information in distinct channels."""

    def __init__(self, input_width, output_width, message_dropout=0.0):
        super().__init__()
        self.own = nn.Linear(input_width, output_width)
        self.incoming = nn.Linear(input_width, output_width, bias=False)
        self.outgoing = nn.Linear(input_width, output_width, bias=False)
        self.norm = nn.LayerNorm(output_width)
        self.message_dropout = float(message_dropout)

    def forward(self, values, incoming, outgoing):
        messages = values
        if self.training and self.message_dropout > 0:
            keep = torch.rand((len(values), 1), device=values.device) >= self.message_dropout
            messages = values * keep / (1.0 - self.message_dropout)
        combined = self.own(values)
        combined = combined + self.incoming(torch.sparse.mm(incoming, messages))
        combined = combined + self.outgoing(torch.sparse.mm(outgoing, messages))
        return F.relu(self.norm(combined))

class TargetSnapshotClassifier(nn.Module):

    def __init__(self, input_width, hidden=48, dropout=0.15, kind='sage', residual=True, message_dropout=0.2):
        super().__init__()
        if kind not in {'sage', 'mlp'}:
            raise ValueError('kind must be sage or mlp')
        self.kind = kind
        self.dropout = float(dropout)
        self.residual = bool(residual and kind == 'sage')
        if kind == 'sage':
            self.first = TargetBidirectionalSageLayer(input_width, hidden, message_dropout)
            self.second = TargetBidirectionalSageLayer(hidden, hidden, message_dropout)
        else:
            self.first = nn.Sequential(nn.Linear(input_width, hidden), nn.LayerNorm(hidden), nn.ReLU())
            self.second = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.output = nn.Linear(hidden, 1)
        if self.residual:
            self.direct = nn.Sequential(nn.Linear(input_width, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, values, incoming=None, outgoing=None):
        if self.kind == 'sage':
            hidden = self.first(values, incoming, outgoing)
            hidden = F.dropout(hidden, self.dropout, self.training)
            hidden = self.second(hidden, incoming, outgoing)
        else:
            hidden = self.first(values)
            hidden = F.dropout(hidden, self.dropout, self.training)
            hidden = self.second(hidden)
        hidden = F.dropout(hidden, self.dropout, self.training)
        logits = self.output(hidden).squeeze(1)
        if self.residual:
            logits = logits + self.direct(values).squeeze(1)
        return logits

def target__snapshot_mean_tensor(TARGET_adjacency, indices):
    matrix = TARGET_adjacency[indices][:, indices].tocsr().astype(np.float32)
    matrix.sum_duplicates()
    matrix.eliminate_zeros()
    degree = np.asarray(matrix.sum(axis=1)).ravel()
    inverse = np.divide(1.0, degree, out=np.zeros_like(degree), where=degree > 0)
    matrix = sp.diags(inverse, format='csr') @ matrix
    coo = matrix.tocoo()
    coordinates = torch.from_numpy(np.vstack([coo.row, coo.col]).astype(np.int64))
    values = torch.from_numpy(coo.data.astype(np.float32, copy=False))
    return torch.sparse_coo_tensor(coordinates, values, size=matrix.shape).coalesce()

def target_fit_transform(profile, cutoff):
    key = f'{profile}_c{cutoff}'
    if key in TARGET_transforms:
        return TARGET_transforms[key]
    cfg = TARGET_PROTOCOL['profiles'][profile]
    visible = TARGET_all_visible if cfg['availability'] == 'all_supplied' else TARGET_target_visible
    fitrows = np.flatnonzero(visible & (TARGET_node_times <= cutoff))
    values = TARGET_X[fitrows].astype(np.float64)
    median = np.nanmedian(values, axis=0)
    assert np.isfinite(median).all()
    values = np.where(np.isfinite(values), values, median)
    keep = np.arange(165)
    corr = np.zeros(165)
    if cfg['filter'] == 'drop90':
        ranked = rankdata(values, axis=0)
        rt = rankdata(TARGET_node_times[fitrows])
        a = ranked - ranked.mean(axis=0)
        b = rt - rt.mean()
        denom = np.sqrt((a * a).sum(axis=0) * (b * b).sum())
        corr = np.divide((a * b[:, None]).sum(axis=0), denom, out=np.zeros(165), where=denom > 0)
        keep = np.flatnonzero(np.abs(corr) < 0.9)
    selected = values[:, keep]
    state = dict(profile=profile, fit_cutoff=cutoff, fit_feature_rows=len(fitrows), fit_feature_indices_sha256=hashlib.sha256(fitrows.tobytes()).hexdigest(), median=median.tolist(), keep=keep.tolist(), time_correlations=corr.tolist(), method=cfg['transform'], clip=8)
    if cfg['transform'] == 'standard':
        mean = selected.mean(axis=0)
        std = selected.std(axis=0)
        std[std < 1e-08] = 1
        state.update(mean=mean.tolist(), std=std.tolist())
    else:
        qt = QuantileTransformer(n_quantiles=min(256, len(selected)), output_distribution='normal', subsample=max(100000, len(selected)), random_state=42).fit(selected)
        state.update(quantiles=qt.quantiles_.tolist(), references=qt.references_.tolist())
    TARGET_save(f'{key}_transform.json', state)
    TARGET_transforms[key] = state
    return state

def target_transform_nodes(state, rows, values_override=None):
    cfg = TARGET_PROTOCOL['profiles'][state['profile']]
    visible = (TARGET_all_visible if cfg['availability'] == 'all_supplied' else TARGET_target_visible)[rows]
    values = TARGET_X[rows] if values_override is None else values_override
    values = np.where(np.isfinite(values), values, np.asarray(state['median']))[:, state['keep']]
    if state['method'] == 'standard':
        result = (values - np.asarray(state['mean'])) / np.asarray(state['std'])
    else:
        qt = QuantileTransformer(output_distribution='normal')
        qt.quantiles_ = np.asarray(state['quantiles'])
        qt.references_ = np.asarray(state['references'])
        qt.n_quantiles_ = len(qt.references_)
        qt.n_features_in_ = len(state['keep'])
        result = qt.transform(values)
    result = np.clip(result, -8, 8)
    result[~visible] = 0
    result = np.column_stack([result, (~visible).astype(np.float32)]).astype(np.float32)
    assert np.isfinite(result).all()
    return result

def target_model_prediction(model, items, q):
    result = np.empty(len(q), np.float64)
    model.eval()
    with torch.no_grad():
        for step in np.unique(TARGET_node_times[q]):
            pos = np.flatnonzero(TARGET_node_times[q] == step)
            item = items[int(step)]
            local = np.searchsorted(item['rows'], q[pos])
            np.testing.assert_array_equal(item['rows'][local], q[pos])
            logits = model(item['x'], *TARGET_adjacency[int(step)])
            result[pos] = torch.sigmoid(logits[torch.from_numpy(local)]).numpy()
    assert np.isfinite(result).all() and ((result >= 0) & (result <= 1)).all()
    return result

def target_fit_one(profile, cutoff, seed, q, stage):
    key = f'{profile}_c{cutoff}_s{seed}'
    if key in TARGET_predictions:
        original, p = TARGET_predictions[key]
        pos = np.searchsorted(original, q)
        np.testing.assert_array_equal(original[pos], q)
        return p[pos]
    started = time.perf_counter()
    assert (TARGET_node_times[q] > cutoff).all() and TARGET_target_visible[q].all()
    fitting_labels = np.where(TARGET_node_times <= cutoff, TARGET_graph_labels, np.nan)
    fitting = np.isfinite(fitting_labels)
    assert not np.isfinite(fitting_labels[TARGET_node_times > cutoff]).any()
    assert fitting.sum() == len(TARGET_query(1, cutoff))
    state = target_fit_transform(profile, cutoff)
    needed = np.unique(np.r_[TARGET_node_times[fitting], TARGET_node_times[q]])
    items = {}
    with threadpool_limits(limits=2):
        torch.set_num_threads(2)
        torch.manual_seed(seed)
        rng = np.random.default_rng(seed)
        for step in needed:
            rows = TARGET_snapshots[int(step)]
            local = np.flatnonzero(fitting[rows])
            items[int(step)] = dict(rows=rows, x=torch.from_numpy(target_transform_nodes(state, rows)), supervised=torch.from_numpy(local), y=torch.from_numpy(fitting_labels[rows[local]].astype(np.float32)))
        model = TargetSnapshotClassifier(len(state['keep']) + 1, hidden=64, dropout=0.15, kind='sage', residual=True, message_dropout=0.2)
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.0001)
        fitsteps = np.unique(TARGET_node_times[fitting])
        mean_count = fitting.sum() / len(fitsteps)
        ratio = float((np.sum(fitting_labels[fitting] == 0) / np.sum(fitting_labels[fitting] == 1)) ** 0.5)
        weight = torch.tensor(ratio, dtype=torch.float32)
        losses = []
        rss = psutil.Process().memory_info().rss / 2 ** 30
        for epoch in range(40):
            model.train()
            loss_total = 0.0
            for step in rng.permutation(fitsteps):
                item = items[int(step)]
                optimizer.zero_grad(set_to_none=True)
                logits = model(item['x'], *TARGET_adjacency[int(step)])[item['supervised']]
                loss = F.binary_cross_entropy_with_logits(logits, item['y'], pos_weight=weight)
                loss = loss * (len(item['y']) / mean_count)
                assert torch.isfinite(loss)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
                loss_total += float(loss.detach())
            losses.append(loss_total / len(fitsteps))
            rss = max(rss, psutil.Process().memory_info().rss / 2 ** 30)
        p = target_model_prediction(model, items, q)
        np.savez_compressed(TARGET_CACHE / f'{key}_weights.npz', **{k: v.detach().numpy() for k, v in model.state_dict().items()})
        np.savez_compressed(TARGET_CACHE / f'{key}_predictions.npz', indices=q, prediction=p)
        reload_model = TargetSnapshotClassifier(len(state['keep']) + 1, hidden=64, dropout=0.15, kind='sage', residual=True, message_dropout=0.2)
        with np.load(TARGET_CACHE / f'{key}_weights.npz') as weights:
            reload_model.load_state_dict({k: torch.from_numpy(weights[k].copy()) for k in weights.files})
        np.testing.assert_array_equal(target_model_prediction(reload_model, items, q), p)
    record = dict(key=key, profile=profile, fit_cutoff=cutoff, seed=seed, stage=stage, epochs=40, threads=2, fitting_labels=int(fitting.sum()), fit_feature_rows=state['fit_feature_rows'], features=len(state['keep']), positive_multiplier=ratio, prediction_rows=len(q), fit_labels_sha256=hashlib.sha256(fitting_labels[fitting].tobytes()).hexdigest(), fit_indices_sha256=hashlib.sha256(np.flatnonzero(fitting).tobytes()).hexdigest(), transform_sha256=TARGET_sha(TARGET_CACHE / f'{profile}_c{cutoff}_transform.json'), weights_sha256=TARGET_sha(TARGET_CACHE / f'{key}_weights.npz'), seconds=time.perf_counter() - started, max_observed_rss_gib=rss, loss_by_epoch=losses, early_stopping=False, reload_all_query_predictions=True)
    TARGET_fits.append(record)
    TARGET_save('fit_log.json', TARGET_fits)
    TARGET_predictions[key] = (q.copy(), p)
    print(f'{key}: {record['seconds']:.1f}s, labels={record['fitting_labels']}, visible fit rows={record['fit_feature_rows']}', flush=True)
    del model, reload_model, optimizer, items
    gc.collect()
    return p

## 2. Data preparation and final ensemble

The raw-data graph builder infers an edge-only node's snapshot only when its
component has exactly one supplied time anchor. IDs and time are not predictors.
Unknown labels never become normal-class targets. Input hashes identify the
original competition files.

The fixed chain is:

1. **V5:** combine the tree ensemble and full-snapshot graph predictions.
2. **V9:**75% rank(V5) +25% rank(mean tabular MLP prediction).
3. **V14:**90% V9 +10% rank(mean availability-matched graph prediction).
4. **V21:**95% V14 +5% rank(mean recency-weighted XGBoost margins), followed by
   a10% incoming-neighbor logit correction.
5. **V22:** three anchored direct-edge iterations at strength0.6, then one
   shared-parent correction at0.2. Six350-tree depth3 XGBoost models learn
   endpoint agreement from past known labels; their102 symmetric features use
  32 past-selected quantile features and log degrees. Mean agreement probabilities
   are converted to positive gates with `max(2*p-1,0)`.

All settings are fixed. This notebook does not perform another model search.

In [5]:
"""Rebuild the fixed V22 submission from the four original competition files.

No earlier predictions, model checkpoints, experiment directories or external
labels are inputs. Generated checkpoints belong only to this reproduction run.
"""
from pathlib import Path
import argparse
import gc
import hashlib
import json
import time
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from scipy.special import expit, logit
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import QuantileTransformer
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier
SEEDS = [42, 52, 62]
FEATURES = [f'feat_{i}' for i in range(1, 166)]
RAW_HASHES = {'train.csv': '6f7e56883f0aa27147262a0d71954ad64702ca57f16fbf9ed58fae21a66dd706', 'test.csv': '344e47c8cd8815f4072d7f6ecbecee8919ab648294a8eb4ed032a026c3eb91db', 'txs_edgelist.csv': 'a35053ba68a98e4382cae2ba65b9d9e36b23b6439e02dff084971b1b72a5156e', 'sample_submission.csv': '3dba74ad6151053aa21f366fb9faf6da7393969a0ba6f8a2ebcec561871f739e'}
EDGE_PARAMS = dict(n_estimators=350, max_depth=3, learning_rate=0.04, min_child_weight=15, reg_lambda=20, reg_alpha=0.5, subsample=0.85, colsample_bytree=0.85, max_bin=128, tree_method='hist', base_score=0.5, n_jobs=2)
RECENT_PARAMS = dict(n_estimators=800, max_depth=5, learning_rate=0.03, min_child_weight=5, reg_lambda=10, reg_alpha=0.1, subsample=0.9, colsample_bytree=0.85, max_bin=128, tree_method='hist', base_score=0.5, n_jobs=2, eval_metric='logloss')

def sha(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

def save(path, value):
    Path(path).write_text(json.dumps(value, indent=2) + '\n')

def rank(values):
    return (rankdata(values, method='average') - 0.5) / len(values)

def load_data(data_dir):
    """Reconstruct same-time topology using features and edges, never labels."""
    raw = Path(data_dir)
    for name, expected in RAW_HASHES.items():
        if not (raw / name).is_file():
            raise FileNotFoundError(f'Place the original competition file at {raw / name}')
        if sha(raw / name) != expected:
            raise ValueError(f'Input hash differs from the competition version: {name}')
    train, test = (pd.read_csv(raw / 'train.csv'), pd.read_csv(raw / 'test.csv'))
    template = pd.read_csv(raw / 'sample_submission.csv')
    edges = pd.read_csv(raw / 'txs_edgelist.csv')
    np.testing.assert_array_equal(template['index'], test['index'])
    assert test['index'].is_unique and len(test) == 15329
    nodes = pd.concat([train[['txId', 'time_step'] + FEATURES], test[['txId', 'time_step'] + FEATURES]], ignore_index=True)
    supplied_ids = pd.Index(nodes.txId)
    endpoints = pd.Index(pd.unique(edges[['txId1', 'txId2']].to_numpy().ravel()))
    all_ids = supplied_ids.append(endpoints.difference(supplied_ids).sort_values())
    source, destination = (all_ids.get_indexer(edges.txId1), all_ids.get_indexer(edges.txId2))
    incoming = sp.csr_matrix((np.ones(len(edges), np.float32), (destination, source)), shape=(len(all_ids), len(all_ids)))
    assert incoming.nnz == len(edges) and (source != destination).all()
    count, component = connected_components(incoming, directed=False)
    low, high = (np.full(count, np.inf), np.full(count, -np.inf))
    np.minimum.at(low, component[:len(nodes)], nodes.time_step)
    np.maximum.at(high, component[:len(nodes)], nodes.time_step)
    assert np.isfinite(low).all() and np.array_equal(low, high)
    times = low[component].astype(np.int32)
    np.testing.assert_array_equal(times[:len(nodes)], nodes.time_step)
    assert np.array_equal(times[source], times[destination])
    x64 = nodes[FEATURES].to_numpy(np.float64)
    graph_x = np.full((len(all_ids), 165), np.nan, np.float32)
    graph_x[:len(nodes)] = x64
    supplied = np.arange(len(all_ids)) < len(nodes)
    labels = np.full(len(all_ids), np.nan)
    labels[:len(train)] = train.label.to_numpy()
    known = np.isfinite(labels)
    assert np.isin(labels[known], [0, 1]).all() and known.sum() == 31235
    target_visible = known.copy()
    target_visible[len(train):len(nodes)] = True
    graph = dict(x=graph_x, node_times=times, supplied_features=supplied, incoming=incoming, outgoing=incoming.T.tocsr())
    return dict(train=train, test=test, template=template, x64=x64, graph=graph, labels=labels, times=times, known=known, target_visible=target_visible, node_ids=all_ids.to_numpy(), source=source, destination=destination)

def sibling_graph(incoming, ids):
    """Connect children of a shared parent with 2..32 visible children."""
    visible = incoming[ids].tocsc()
    degree = np.asarray(visible.sum(axis=0)).ravel()
    keep = (degree >= 2) & (degree <= 32)
    visible = visible[:, keep]
    relation = (visible @ sp.diags(1 / (degree[keep] - 1)) @ visible.T).tocsr()
    relation.setdiag(0)
    relation.eliminate_zeros()
    return relation

def pair_features(ids, source, destination, values, degrees):
    a, b = (values[ids[source]], values[ids[destination]])
    da, db = (degrees[ids[source]], degrees[ids[destination]])
    return np.column_stack([np.abs(a - b), np.abs(da - db), np.minimum(a, b), np.maximum(a, b), np.minimum(da, db), np.maximum(da, db)]).astype(np.float32)

def train_predict(context, cutoff, query, output_dir):
    """Fit the frozen final recipe at one historical or final cutoff.

    Historical evaluation is retrospective: recipe choices have already used
    these time periods. It is not a fresh holdout or the original adaptive CV.
    """
    global BASE_CACHE, BASE_MODEL_TAG, TARGET_CACHE, TARGET_PROTOCOL, TARGET_X, TARGET_adjacency, TARGET_all_visible, TARGET_fits, TARGET_graph_labels, TARGET_node_times, TARGET_predictions, TARGET_query, TARGET_save, TARGET_sha, TARGET_snapshots, TARGET_target_visible, TARGET_transforms
    started = time.perf_counter()
    cache = Path(output_dir) / f'c{cutoff}'
    cache.mkdir(parents=True, exist_ok=True)
    query = np.asarray(query, dtype=int)
    y, times, graph, x64 = (context[k] for k in ['labels', 'times', 'graph', 'x64'])
    fit = np.flatnonzero(context['known'] & (times <= cutoff))
    assert len(query) and np.all(times[query] > cutoff) and len(fit)
    identity = dict(cutoff=cutoff, query_sha256=hashlib.sha256(query.tobytes()).hexdigest(), raw_hashes=RAW_HASHES, seeds=SEEDS)
    if (cache / 'inputs.json').exists():
        assert json.loads((cache / 'inputs.json').read_text()) == identity
    else:
        save(cache / 'inputs.json', identity)
    BASE_CACHE = cache
    logs = []

    def component(tag, function):
        global BASE_MODEL_TAG
        BASE_MODEL_TAG = tag
        path = cache / f'{tag}_prediction.npy'
        if path.exists():
            return np.load(path)
        prediction, _, info = function()
        np.save(path, prediction)
        save(cache / f'{tag}_fit.json', {k: v for k, v in info.items() if k != 'history'})
        logs.append(tag)
        print(f'Completed {tag}', flush=True)
        return prediction
    runner = BaseTreeRunner(x64, np.empty((len(x64), 0)), x64, x64, y[:len(x64)], times[:len(x64)], cache)
    with threadpool_limits(limits=2):
        e04 = component('E04', lambda: runner.fit(dict(id='E04', reference='E04'), cutoff, query, rounds=400, stage='fixed_reproduction'))
        cats = []
        for policy, rounds in [('raw', 336), ('drop', 244)]:
            for seed in SEEDS:
                cats.append(component(f'CAT_{policy}_s{seed}', lambda policy=policy, rounds=rounds, seed=seed: runner.fit(dict(id=f'CAT_{policy}', reference='CAT', policy=policy), cutoff, query, seed=seed, rounds=rounds, stage='fixed_reproduction')))
        v4 = 0.5 * e04 + 0.5 * np.mean(cats, axis=0)
        sage = []
        for seed in SEEDS:
            sage.append(component(f'SAGE_s{seed}', lambda seed=seed: base_fit_predict_graph_model(graph, np.where(times <= cutoff, y, np.nan), cutoff, query, dict(kind='sage', hidden=64, dropout=0.15, message_dropout=0.2, learning_rate=0.003, weight_decay=0.0001, class_power=0.5, max_epochs=120, patience=20, min_epochs=8, threads=2), seed=seed, epochs=40)))
        v5 = 0.75 * v4 + 0.25 * np.mean(sage, axis=0)
        policy = BaseTemporalFeatureTransform('drop90').fit(x64[fit], times[fit])
        joblib.dump(policy, cache / 'C37_feature_policy.joblib')
        a, b = (policy.transform(x64[fit]), policy.transform(x64[query]))
        neural = []
        for seed in SEEDS:
            neural.append(component(f'C37_s{seed}', lambda seed=seed: base_fit_predict_ensemble(a, y[fit], b, dict(k=8, width=128, preprocessing='quantile', class_power=0.0, learning_rate=0.001, weight_decay=0.0001, batch_size=512, max_epochs=80, patience=12, min_epochs=8, threads=2, dropout=0.05), seed=seed, epochs=24)))
        c37 = np.mean(neural, axis=0)
        v9 = 0.75 * rank(v5) + 0.25 * rank(c37)
        TARGET_CACHE = cache
        TARGET_save = lambda name, value: save(cache / name, value)
        TARGET_sha = sha
        TARGET_X = graph['x']
        TARGET_node_times = times
        TARGET_graph_labels = y
        TARGET_all_visible = graph['supplied_features']
        TARGET_target_visible = context['target_visible']
        TARGET_query = lambda lo, hi: np.flatnonzero(context['known'] & (times >= lo) & (times <= hi))
        TARGET_PROTOCOL = dict(profiles={'TARGET_QNT': dict(availability='targets', filter='drop90', transform='quantile')})
        TARGET_transforms = {}
        TARGET_predictions = {}
        TARGET_fits = []
        TARGET_snapshots = {int(t): np.flatnonzero(times == t) for t in np.unique(times)}
        TARGET_adjacency = {t: (target__snapshot_mean_tensor(graph['incoming'], rows), target__snapshot_mean_tensor(graph['outgoing'], rows)) for t, rows in TARGET_snapshots.items()}
        target_parts = []
        for seed in SEEDS:
            path = cache / f'TARGET_QNT_c{cutoff}_s{seed}_predictions.npz'
            if path.exists():
                stored = np.load(path)
                np.testing.assert_array_equal(stored['indices'], query)
                part = stored['prediction']
            else:
                part = target_fit_one('TARGET_QNT', cutoff, seed, query, 'fixed_reproduction')
                logs.append(f'TARGET_QNT_s{seed}')
            target_parts.append(part)
        v14 = 0.9 * v9 + 0.1 * rank(np.mean(target_parts, axis=0))
        x = x64.astype(np.float32)
        corr = pd.DataFrame(x[fit]).corrwith(pd.Series(times[fit]), method='spearman').fillna(0).to_numpy()
        columns = np.flatnonzero(np.abs(corr) < 0.9)
        weights = 2.0 ** (-(cutoff - times[fit]) / 10)
        weights /= weights.mean()
        recent = []
        selector = None
        for seed in SEEDS:
            model = XGBClassifier(**RECENT_PARAMS, random_state=seed)
            path = cache / f'RECENT_s{seed}.ubj'
            if path.exists():
                model.load_model(path)
            else:
                model.fit(x[np.ix_(fit, columns)], y[fit].astype(int), sample_weight=weights)
                model.save_model(path)
                logs.append(f'RECENT_s{seed}')
            recent.append(model.predict(x[np.ix_(query, columns)], output_margin=True))
            if seed == 42:
                selector = model
        incoming = graph['incoming'].astype(np.float64)
        direct = incoming[query][:, query].tocsr()
        degree = np.asarray(direct.sum(axis=1)).ravel()
        anchor = 0.95 * v14 + 0.05 * rank(np.mean(recent, axis=0))
        margins = logit(np.clip(anchor, 1e-08, 1 - 1e-08))
        parent = np.divide(direct @ margins, degree, out=margins.copy(), where=degree > 0)
        v21 = expit(0.9 * margins + 0.1 * parent)
        selected = columns[np.argsort(selector.feature_importances_)[-32:]]
        qt = QuantileTransformer(n_quantiles=min(1000, len(fit)), output_distribution='normal', subsample=None, random_state=42).fit(x[np.ix_(fit, selected)])
        transformed = np.clip(qt.transform(x[:, selected]), -4, 4)
        degrees = np.log1p(np.column_stack([np.asarray(incoming.sum(axis=1)).ravel(), np.asarray(incoming.sum(axis=0)).ravel()]))
        joblib.dump(dict(quantiles=qt, columns=selected, fit_indices=fit), cache / 'edge_transform.joblib')
        np.savez_compressed(cache / 'recent_state.npz', indices=fit, columns=columns, weights=weights)
        predictions = {}
        for kind in ['direct', 'siblings']:
            fit_graph = incoming[fit][:, fit] if kind == 'direct' else sp.triu(sibling_graph(incoming, fit), k=1)
            f = fit_graph.tocoo()
            src, dst = (f.col, f.row)
            labels = y[fit].astype(int)
            same = labels[src] == labels[dst]
            if kind == 'direct':
                cells = 2 * labels[src] + labels[dst]
                weight = 1 / np.bincount(cells, minlength=4)[cells]
            else:
                cells = np.where(same, labels[src], 2)
                totals = np.bincount(cells, weights=f.data, minlength=3)
                weight = f.data * np.array([0.25, 0.25, 0.5])[cells] / totals[cells]
            weight /= weight.mean()
            fm = pair_features(fit, src, dst, transformed, degrees)
            relation = (direct if kind == 'direct' else sibling_graph(incoming, query)).tocoo()
            qm = pair_features(query, relation.col, relation.row, transformed, degrees)
            parts = []
            for seed in SEEDS:
                model = XGBClassifier(**EDGE_PARAMS, random_state=seed)
                path = cache / f'{kind}_s{seed}.ubj'
                if path.exists():
                    model.load_model(path)
                else:
                    model.fit(fm, same.astype(int), sample_weight=weight)
                    model.save_model(path)
                    logs.append(f'{kind}_s{seed}')
                parts.append(model.predict_proba(qm)[:, 1])
            probability = np.mean(parts, axis=0)
            weighted = sp.csr_matrix((relation.data * np.maximum(2 * probability - 1, 0), (relation.row, relation.col)), shape=relation.shape)
            if kind == 'direct':
                weighted = weighted + weighted.T
                count = np.asarray(direct.sum(axis=1)).ravel() + np.asarray(direct.sum(axis=0)).ravel()
                own = logit(np.clip(v21, 1e-08, 1 - 1e-08))
                current = own.copy()
                mass = np.asarray(weighted.sum(axis=1)).ravel()
                for _ in range(3):
                    current = own + 0.6 * (weighted @ current - mass * own) / np.maximum(count, 1)
                risk = expit(current)
            else:
                count = np.asarray(relation.tocsr().sum(axis=1)).ravel()
                own = logit(np.clip(risk, 1e-08, 1 - 1e-08))
                risk = expit(own + 0.2 * (weighted @ own - np.asarray(weighted.sum(axis=1)).ravel() * own) / np.maximum(count, 1))
            predictions[kind] = risk.copy()
            np.savez_compressed(cache / f'{kind}_edges.npz', indices=query, source=relation.col, destination=relation.row, multiplicity=relation.data, seed_probabilities=np.array(parts), probability=probability)
        assert np.isfinite(risk).all() and ((risk >= 0) & (risk <= 1)).all()
        np.savez_compressed(cache / 'predictions.npz', indices=query, E04=e04, V4=v4, V5=v5, C37=c37, V9=v9, GRAPH=np.mean(target_parts, axis=0), V14=v14, V21=v21, V22=risk)
        summary = dict(cutoff=cutoff, known_fit_rows=len(fit), query_rows=len(query), seeds=SEEDS, seconds=time.perf_counter() - started, raw_hashes=RAW_HASHES, new_model_fits_this_invocation=len(logs), validation='Retrospective fixed final recipe on reused periods; not original adaptive metrics or an untouched holdout.')
        if np.isfinite(y[query]).all():
            summary['auc'] = float(roc_auc_score(y[query], risk))
        save(cache / 'summary.json', summary)
        print(json.dumps(summary, indent=2), flush=True)
    gc.collect()
    return (risk, summary)

def generate_submission(context, output_dir):
    query = np.arange(len(context['train']), len(context['train']) + len(context['test']))
    risk, summary = train_predict(context, 35, query, output_dir)
    frame = pd.DataFrame({'index': context['test']['index'].to_numpy(), 'target': risk})
    np.testing.assert_array_equal(frame['index'], context['template']['index'])
    path = Path(output_dir) / 'submission_edgegate_v22_reproduced.csv'
    frame.to_csv(path, index=False, float_format='%.17g')
    reloaded = pd.read_csv(path, float_precision='round_trip')
    assert list(reloaded.columns) == ['index', 'target'] and reloaded['index'].is_unique
    np.testing.assert_array_equal(reloaded['index'], frame['index'])
    np.testing.assert_array_equal(reloaded.target, risk)
    save(Path(output_dir) / 'submission_checks.json', dict(passed=True, rows=len(frame), columns=list(frame.columns), sha256=sha(path), minimum=float(risk.min()), maximum=float(risk.max()), template_order_exact=True, csv_roundtrip_exact=True))
    return (path, summary)

## 3. Load and inspect the original data

The first complete run trains25 final models plus25 models for the chronological
diagnostic. On the tested CPU these stages took approximately7 and6 minutes,
respectively; hardware may differ. At most two numerical threads are used.
Generated checkpoints stay under `outputs/v22_standalone/` for reuse. Use a new
output directory if changing code or settings. No network or Kaggle upload is
performed.

In [6]:
from IPython.display import display

ROOT = Path.cwd()
DATA = ROOT / "data/raw"
OUTPUT = ROOT / "outputs/v22_standalone"
RUN_VALIDATION = True
OUTPUT.mkdir(parents=True, exist_ok=True)
context = load_data(DATA)
train, test = context["train"], context["test"]
display(pd.Series({
    "training_rows": len(train), "known_labels": int(train.label.notna().sum()),
    "normal": int((train.label == 0).sum()), "illicit": int((train.label == 1).sum()),
    "unknown_labels": int(train.label.isna().sum()), "test_rows": len(test),
    "graph_nodes": len(context["node_ids"]), "directed_edges": len(context["source"]),
    "training_steps": "1–35", "test_steps": "36–49",
}))

training_rows     141772
known_labels       31235
normal             27591
illicit             3644
unknown_labels    110537
test_rows          15329
graph_nodes       203769
directed_edges    234355
training_steps      1–35
test_steps         36–49
dtype: object

## 4. Chronological validation

Fit through step30; evaluate known labels in steps31–35. No query labels enter
fitting or preprocessing. This is a **retrospective fixed-recipe diagnostic**,
distinct from the originally adaptive upstream validation. These periods were
repeatedly reused during development; they are not an untouched holdout.
The verified fixed-recipe AUC is approximately0.998650661. It is not a public
score forecast. Set `RUN_VALIDATION=False` to generate the final CSV only.

In [7]:
if RUN_VALIDATION:
    ids = np.flatnonzero(context["known"] & (context["times"] > 30) & (context["times"] <= 35))
    validation_prediction, validation_summary = train_predict(context, 30, ids, OUTPUT)
    display(pd.Series(validation_summary).drop("raw_hashes"))
else:
    print("Chronological diagnostic skipped; final recipe remains unchanged.")

TARGET_QNT_c30_s42: 24.8s, labels=26905, visible fit rows=26905
{
  "cutoff": 30,
  "known_fit_rows": 26905,
  "query_rows": 4330,
  "seeds": [
    42,
    52,
    62
  ],
  "seconds": 27.91060312499758,
  "raw_hashes": {
    "train.csv": "6f7e56883f0aa27147262a0d71954ad64702ca57f16fbf9ed58fae21a66dd706",
    "test.csv": "344e47c8cd8815f4072d7f6ecbecee8919ab648294a8eb4ed032a026c3eb91db",
    "txs_edgelist.csv": "a35053ba68a98e4382cae2ba65b9d9e36b23b6439e02dff084971b1b72a5156e",
    "sample_submission.csv": "3dba74ad6151053aa21f366fb9faf6da7393969a0ba6f8a2ebcec561871f739e"
  },
  "new_model_fits_this_invocation": 1,
  "validation": "Retrospective fixed final recipe on reused periods; not original adaptive metrics or an untouched holdout.",
  "auc": 0.9986506609332696
}


cutoff                                                                           30
known_fit_rows                                                                26905
query_rows                                                                     4330
seeds                                                                  [42, 52, 62]
seconds                                                                   27.910603
new_model_fits_this_invocation                                                    1
validation                        Retrospective fixed final recipe on reused per...
auc                                                                        0.998651
dtype: object

## 5. Final training and submission

Fit on all known labels through step35 and predict test steps36–49. The output
uses the literal test `index`, exact template order and finite continuous scores
in[0,1]. The saved CSV is reloaded and checked. There is no dependence on an old
prediction CSV or a pretrained model; checkpoints are generated by this workflow.

In [8]:
submission_path, final_summary = generate_submission(context, OUTPUT)
display(pd.Series(final_summary).drop("raw_hashes"))
print("Submission:", submission_path.relative_to(ROOT))
display(pd.Series(json.loads((OUTPUT / "submission_checks.json").read_text())))

{
  "cutoff": 35,
  "known_fit_rows": 31235,
  "query_rows": 15329,
  "seeds": [
    42,
    52,
    62
  ],
  "seconds": 3.9851622920105,
  "raw_hashes": {
    "train.csv": "6f7e56883f0aa27147262a0d71954ad64702ca57f16fbf9ed58fae21a66dd706",
    "test.csv": "344e47c8cd8815f4072d7f6ecbecee8919ab648294a8eb4ed032a026c3eb91db",
    "txs_edgelist.csv": "a35053ba68a98e4382cae2ba65b9d9e36b23b6439e02dff084971b1b72a5156e",
    "sample_submission.csv": "3dba74ad6151053aa21f366fb9faf6da7393969a0ba6f8a2ebcec561871f739e"
  },
  "new_model_fits_this_invocation": 0,
  "validation": "Retrospective fixed final recipe on reused periods; not original adaptive metrics or an untouched holdout."
}


cutoff                                                                           35
known_fit_rows                                                                31235
query_rows                                                                    15329
seeds                                                                  [42, 52, 62]
seconds                                                                    3.985162
new_model_fits_this_invocation                                                    0
validation                        Retrospective fixed final recipe on reused per...
dtype: object

Submission: outputs/v22_standalone/submission_edgegate_v22_reproduced.csv


passed                                                               True
rows                                                                15329
columns                                                   [index, target]
sha256                  feaff00f7cf88b3be6e4b887d5a8b01b90b7eb3ab52940...
minimum                                                          0.007838
maximum                                                          0.998421
template_order_exact                                                 True
csv_roundtrip_exact                                                  True
dtype: object

## 6. Results and limitations

The original V22 public ROC-AUC is **0.95716**. Its CSV SHA256 is
`25dcee6c22846a2937bd00ff9dc118afcb4728ae571423621cebf560e6995d21`.
Fresh raw-data training of this fixed implementation reproduced the original
rank order and ties exactly; maximum score difference was8.33e-16 on the tested
machine. Floating-point details can vary across platforms.

Local gains did not reliably transfer to Kaggle. The validation windows were
adaptively reused, and the graph assumes same-snapshot availability. These
limitations belong in the report. The separate PDF report should link to the
GitHub repository; running this notebook does not submit the report or upload
the generated CSV.